# 03 - Fine-tune detection models

PRD section 3.8, flow 4. For every model in `config.yaml` (`detection.models`) we fine-tune five times, once per mode:

| Mode | Training data |
|---|---|
| 1 | real tune images only |
| 2 | real + method A (objects added with inpainting) |
| 3 | real + method B (capture conditions changed with InstructPix2Pix) |
| 4 | real + A + B (half of each) |
| 5 | real + classical augmentation (dark, blur, noise, bright) of the same source images as mode 3 |

**Fairness (PRD 3.9):** modes 2-5 add the same number of extra images (`training.synthetic_per_mode`), so a gain cannot come
from more data. All runs share one recipe (`training:` in the config); only the data changes.

Course tools: `AutoImageProcessor` / `AutoModelForObjectDetection` with `id2label`, frozen backbone (`requires_grad_(False)`),
Hugging Face `Trainer` with `fp16`, `dataloader_drop_last`, and early stopping with patience 10 on the control-split loss.
Finished runs are skipped and an interrupted run resumes from its last checkpoint, so you can rerun after a disconnect.

In [ ]:
%pip install -q transformers accelerate timm

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch

import os, subprocess, sys, yaml
REPO = "/content/genai-shelter-validator"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))
sys.path.insert(0, f"{REPO}/Code")

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
os.makedirs(ROOT, exist_ok=True)
print("Config loaded, Drive ready at", ROOT)

## 1. Load real splits (tune / val / test) and synthetic images

Only the **tune** images and the filtered synthetic images are used for training, the **val** (control) images decide when to
stop, and the **test** images are not touched in this notebook. Images are copied from Drive to the Colab disk once, which
makes the training reads fast.

In [ ]:
import gc, json, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint
import common as C

device = C.device
det, tr = cfg["detection"], cfg["training"]
table = C.load_objects(REPO, cfg)
classes = table["name"].tolist()
C.check_models(det["models"])
assert len(det["models"]) >= 3, "PRD 3.8: at least 3 detection models in config.yaml"

real = C.split_real(C.load_real(ROOT, classes), cfg, ROOT)
display(C.split_table(real, classes))
duplicates = C.find_duplicates(real)
if duplicates:
    print("WARNING: the same image appears in different splits (fix before trusting the results):", duplicates)

tune = C.cache_local([r for r in real if r["split"] == "tune"])
val = C.cache_local([r for r in real if r["split"] == "val"])
synthetic = C.cache_local(C.load_synthetic(ROOT, classes))          # passed the filter of notebook 02 only
syn_a = [r for r in synthetic if r["method"] == "A"]
syn_b = [r for r in synthetic if r["method"] == "B"]
modes = C.build_modes(tune, syn_a, syn_b, tr["synthetic_per_mode"], cfg["seed"], cfg["generation"]["image_size"])

summary = pd.DataFrame({C.MODES[m]: {"images": len(recs), **{k: sum(x["kind"] == k for x in recs) for k in ("real", "synthetic", "classical")}}
                        for m, recs in modes.items()}).T
display(summary)
print("control (val) images:", len(val))

## 2. For each model in the config, fine-tune on the 5 setups

One recipe for everything: AdamW (Trainer default), the learning rate / weight decay / epochs of `training:` in the config,
`fp16` on GPU, backbone frozen. The learning rate is a starting value, not tuned (no time on a T4); if all runs look
undertrained, raise `training.epochs` or `learning_rate` for every run, not for one.

In [ ]:
def run_one(model_id, mode):
    tag = model_id.split("/")[-1]
    out = f"{ROOT}/results/models/{tag}/mode{mode}"
    ckpt = f"{ROOT}/results/checkpoints/{tag}/mode{mode}"
    if os.path.exists(f"{out}/run.json"):
        print(f"skip {tag} mode {mode}: already finished")
        return
    set_seed(cfg["seed"])
    model, processor = C.load_detector(model_id, classes, det["image_size"], det["freeze_backbone"])
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = TrainingArguments(
        output_dir=ckpt,
        remove_unused_columns=False,                    # keep pixel_values and labels
        per_device_train_batch_size=det["batch_size"],
        per_device_eval_batch_size=1,                   # the control split is tiny; batch 1 keeps every image
        num_train_epochs=tr["epochs"],
        learning_rate=tr["learning_rate"],
        weight_decay=tr["weight_decay"],
        max_grad_norm=0.1,                              # DETR-style gradient clipping
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,                    # best control loss, not the last epoch
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        prediction_loss_only=True,
        logging_strategy="epoch",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=min(4, os.cpu_count() or 1),
        dataloader_pin_memory=True,
        dataloader_drop_last=True,
        seed=cfg["seed"], data_seed=cfg["seed"],
        report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=C.DetectionDataset(modes[mode], processor),
                      eval_dataset=C.DetectionDataset(val, processor), data_collator=C.collate_fn,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=tr["patience"])])
    resume = get_last_checkpoint(ckpt) if os.path.isdir(ckpt) else None
    print(f"=== {tag} | mode {mode} ({C.MODES[mode]}) | {len(modes[mode])} train images | "
          f"{n_trainable / 1e6:.1f}M trainable parameters | resume: {resume}")
    try:
        trainer.train(resume_from_checkpoint=resume)
    except torch.cuda.OutOfMemoryError:
        raise RuntimeError("GPU out of memory. Lower detection.batch_size in config.yaml and rerun "
                           "(finished runs are kept).") from None
    trainer.save_model(out)                             # the best checkpoint
    processor.save_pretrained(out)
    json.dump(trainer.state.log_history, open(f"{out}/log_history.json", "w"))
    json.dump({"model": model_id, "mode": mode, "mode_name": C.MODES[mode], "train_images": len(modes[mode]),
               "val_images": len(val), "best_val_loss": trainer.state.best_metric, "epochs_run": trainer.state.epoch,
               "seed": cfg["seed"], "training": tr, "detection": det}, open(f"{out}/run.json", "w"), indent=1)
    shutil.rmtree(ckpt, ignore_errors=True)             # optimizer states are big; the final model is saved
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


for model_id in det["models"]:
    for mode in C.MODES:
        run_one(model_id, mode)

## 3. Use the val split to decide when to stop; loss curves

Early stopping watched the control-split loss (patience from the config). The curves below come from the saved logs.
A control loss that is still falling at the last epoch means the run was cut by `training.epochs`, not by early stopping.

In [ ]:
colors = {1: "#2a78d6", 2: "#eb6834", 3: "#1baf7a", 4: "#eda100", 5: "#e87ba4"}
os.makedirs(f"{ROOT}/results/plots", exist_ok=True)
for model_id in det["models"]:
    tag = model_id.split("/")[-1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for mode in C.MODES:
        path = f"{ROOT}/results/models/{tag}/mode{mode}/log_history.json"
        if not os.path.exists(path):
            continue
        log = json.load(open(path))
        for ax, key in zip(axes, ["loss", "eval_loss"]):
            points = [(e["epoch"], e[key]) for e in log if key in e]
            ax.plot(*zip(*points), color=colors[mode], label=C.MODES[mode], linewidth=2)
    for ax, title in zip(axes, ["training loss", "control (val) loss"]):
        ax.set_title(f"{tag}: {title}")
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.2)
    axes[0].legend(frameon=False)
    plt.tight_layout()
    plt.savefig(f"{ROOT}/results/plots/loss_{tag}.png", dpi=200)
    plt.show()